# Check-in 3: the core watermark benchmark
CMPE 261, Group 6. TR, RI, SFW, GS, WIND, PRC on pinned SD 2.1-base.

This notebook prepares a fresh dataset and retains historical runs unchanged. Full runs require a passing six-scheme GPU preflight and immutable inputs, environment, and key state. No sample counts, attack strengths, or budgets are reduced automatically.

Run each cell in order. The full queue is serial. Do not run a second worker against the same output directory. Interrupting an experiment is safe; reopen this notebook and rerun the same command with the same source and environment to resume.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/cmpe261-watermark-forgery')
assert DRIVE_ROOT.is_dir(), 'Use the existing project Drive folder; do not create a replacement.'
OUT = DRIVE_ROOT / 'checkin3'
PILOT = DRIVE_ROOT / 'checkin3-preflight-v1'
SOURCE_ZIP = Path('/content/checkin3-source.zip')
REPO = Path('/content/cmpe261-watermark-forgery-checkin3')
BRANCH = 'checkin3-core-benchmark'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))
print('Drive:', DRIVE_ROOT, '| full results:', OUT, '| independent pilot:', PILOT)


## Source
Before publication, upload the supplied `checkin3-source.zip` through Colab's Files pane. After the branch is published, the clone path below can be used instead. Never merge with old Check-in 1 or Check-in 2 result directories. Review the source revision printed here before proceeding.


In [ ]:
import zipfile
if SOURCE_ZIP.exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Source already exists. Keep it unchanged when resuming; skip this cell.')
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        archive.extractall(REPO)
else:
    if not REPO.exists():
        subprocess.run(['git','clone','--branch',BRANCH,
                        'https://github.com/yashashav-dk/cmpe261-watermark-forgery.git',str(REPO)],check=True)
assert (REPO/'src/wmforge/checkin3.py').exists(), 'Upload the Check-in 3 source archive.'
os.chdir(REPO)
print('Source:', REPO)


## Isolated Python 3.12 and recorded GPU stack
The notebook kernel may use another Python version. All experiments below run in a separate Python 3.12 environment. Setup uses checked subprocess calls and stops on errors. The complete compatible installed environment is captured after `pip check`; transitive versions are then bound into the run manifest. Reuse the captured environment for a disconnected worker.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','python','install','3.12'],check=True)
PY = Path('/content/checkin3-py312/bin/python')
if not PY.exists():
    subprocess.run([sys.executable,'-m','uv','venv','--seed','--python','3.12',str(PY.parent.parent)],check=True)
subprocess.run([str(PY),'-m','pip','install','torch==2.10.0','torchvision==0.25.0',
                '--index-url','https://download.pytorch.org/whl/cu128'],check=True)
subprocess.run([str(PY),'-m','pip','install','-r','requirements-checkin3.txt'],check=True)
subprocess.run([str(PY),'-m','pip','install','-e','.[dev]'],check=True)
subprocess.run([str(PY),'-m','pip','check'],check=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONPATH'] = str(REPO/'src')
sys.path.insert(0,str(REPO/'src'))
subprocess.run([str(PY),'-c','from wmforge.checkin3 import check_environment; print(check_environment())'],check=True)
freeze = subprocess.check_output([str(PY),'-m','pip','freeze'],text=True)
setup = DRIVE_ROOT/'checkin3-setup'
setup.mkdir(exist_ok=True)
if (setup/'environment.txt').exists():
    assert (setup/'environment.txt').read_text()==freeze, 'Environment differs from saved setup; restore its exact freeze before resuming.'
else:
    (setup/'environment.txt').write_text(freeze)
(setup/'python.txt').write_text(subprocess.check_output([str(PY),'--version'],text=True))
(setup/'gpu.txt').write_text(subprocess.check_output(['nvidia-smi'],text=True))
print(freeze)


## Tests and deterministic input extension
Select 200 unique prompts and 100 unique COCO photographs with seed 261. Preserve the committed 50-item selections as exact prefixes, including the first five forgery covers. Selection refuses to overwrite different existing files. Save the selections on Drive and commit these exact files with the measurements after the run.


In [ ]:
subprocess.run([str(PY),'-m','pytest','-q','-m','not slow'],check=True)
subprocess.run([str(PY),'scripts/select_checkin3_data.py'],check=True)
import shutil
for name in ['prompts_checkin3.txt','coco_val2017_checkin3.txt','checkin3-selection.json']:
    source=REPO/'data'/name
    destination=setup/name
    if destination.exists():
        assert destination.read_bytes()==source.read_bytes(), 'Committed selections changed.'
    else:
        shutil.copy2(source,destination)
print('Selections saved; preserve and version these files.')


## GPU preflight and measured cost
This pilot is separate from the benchmark. It exercises all six schemes, four removals, score finiteness, key restoration, image dimensions, LPIPS, 10 imprint steps, checkpoint replay, GPU memory, and Drive save/reopen. A pilot directory containing evidence is immutable: inspect an existing successful preflight rather than rerunning it. Choose a new numbered pilot directory only for a deliberately new pilot.

Inspect `preflight.json` and actual GPU cost before the full queue. The estimated imprint hours exclude baseline and removal cost; the historical 16–20-hour total on an RTX PRO 6000 is a planning estimate only. A T4 may have insufficient memory for the full-precision workload. Failed preflight blocks full execution.


In [ ]:
if (PILOT/'preflight.json').exists():
    import json
    pilot=json.loads((PILOT/'preflight.json').read_text())
    print(json.dumps(pilot,indent=2))
    assert pilot['passed'], 'Inspect failure; fix the environment, then use a new pilot directory.'
else:
    subprocess.run([str(PY),'-m','wmforge.preflight','--out',str(PILOT)],check=True)


## Full F1, F2, F3
F1: 100 watermarked + 200 unwatermarked generations + 100 real photos per scheme. Inclusion uses the native detector only: clean TPR ≥90%, generated FPR ≤5%.

Calibration uses only the 200 unwatermarked scores, oriented so larger values are stronger evidence, with a strict threshold allowing at most two positives. This is an empirical operating point, with 0.5-percentage-point resolution. PRC's score-only calibrated test is separate from its native decoder verdict.

F2: same 10 covers per included scheme, 150 steps per cover, checks every 10, no stopping at detection. Primary rates and PSNR/SSIM/LPIPS use the saved images at steps 50/100/150.

F3: JPEG quality 95, Gaussian blur radius 2, centered crop ratio 0.8 and resize, and SD 2.1-base purification strength 0.3 with empty prompt. Apply to every one of the 100 saved watermarked images, including initial detector misses. All decisions use saved artifacts. Never run these cells concurrently.


In [ ]:
BASE = [str(PY),'-m','wmforge.checkin3','--out',str(OUT),
        '--preflight',str(PILOT/'preflight.json')]
subprocess.run(BASE+['--stages','baseline'],check=True)


In [ ]:
subprocess.run(BASE+['--stages','imprint'],check=True)


In [ ]:
subprocess.run(BASE+['--stages','removal'],check=True)


## Progress, reporting, backups
This cell can read an atomically published log from another notebook while the worker runs. Do not start a second writer. Incomplete cells are marked as provisional. Continuous metrics use 10,000 bootstrap resamples; binary rates use Wilson 95% intervals. Results cannot be pooled with historical stop-at-first-detection runs.


In [ ]:
subprocess.run([str(PY),'-m','wmforge.checkin3_report','--run',str(OUT)],check=True)
from IPython.display import Markdown,display
display(Markdown((OUT/'summary-v2.md').read_text()))


Run backups only after the worker has stopped, so the archive contains a consistent set of log, checkpoints, keys, and measured images. Keep key material in the team Drive; publish only non-secret manifests, selections, results, and report artifacts.


In [ ]:
from datetime import datetime,timezone
# A nonblocking lock refuses to archive during an active write.
from wmforge.checkin3 import single_writer
with single_writer(OUT):
    backup=DRIVE_ROOT/('checkin3-backup-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    shutil.make_archive(str(backup),'zip',root_dir=OUT)
print('Backup:',str(backup)+'.zip')


## Draft report and final checks
Generate the draft with the verified summary, inspect every PDF page, confirm member contributions and instructor/TA access, and review partial work explicitly. The report generator labels absent results as pending. Repository publication and Canvas submission are separate actions.


In [ ]:
subprocess.run([str(PY),'scripts/checkin3_draft.py','--run',str(OUT),
                '--out',str(OUT/'Group6_CMPE261_CheckIn_3_draft.pdf')],check=True)
print('Draft:',OUT/'Group6_CMPE261_CheckIn_3_draft.pdf')
